In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))
import jax
import jax.numpy as jnp
import pjax
from pjax import nn, optim
import matplotlib.pyplot as plt
from experiments.shared.data import MNISTDataModule

# Configure logging
def log(msg):
    print(f"[INFO] {msg}")

# Load Data
batch_size = 1024
dataset = MNISTDataModule(batch_size=batch_size)
train_data = dataset.train_dataloader()
test_data = dataset.test_dataloader()

# 1. Define the model
class CNN_pjax(nn.Module):
    """PJAX CNN model with optional skip connections and max pooling.

    Args:
        hidden_features: list of integers specifying channel sizes for conv layers.
        in_features: number of input channels.
        size_2d: spatial size of square input images (height = width).
        classes: number of output classes for classification.
        skip: whether to concatenate features from all conv layers.
        max_pool: whether to apply max pooling over spatial dimensions.
        stride: convolution stride for all layers.
    """

    def __init__(self, classes):
        super().__init__()
        self.conv_layer = nn.FftConv2D(28,28,1,1, 3)
        self.relu = nn.ReLU(2*28)

        self.out = nn.Linear(28*28, classes)

    def __call__(self, x):
        xs = []
        x = self.conv_layer(x)
        # jax.debug.print("x after conv {}", x)
        x= pjax.reshape(x, (x.shape[0], -1))
        return self.out(x)

# 2. Initialize model and optimizer
key = jax.random.key(0)
model = CNN_pjax( classes=10)
params = model.init(key)

optimizer = optim.AlternatingProjections(steps_per_update=50)

# 3. Define steps
@jax.jit
def train_step(params, x, y):
    def apply_fn(params):
        logits = model.apply(params, x)
        y_one_hot = jax.nn.one_hot(y, num_classes=10)
        y_one_hot = y_one_hot.astype(jax.numpy.complex64)
        return pjax.means_squared_error(logits, y_one_hot)

    updated_params, loss = optimizer.update(apply_fn, params)
    return updated_params, loss

@jax.jit
def eval_step(params, x, y):
    logits = model.apply(params, x)
    if jnp.iscomplexobj(logits):
        predictions = jnp.argmax(logits.real, axis=-1)
    else:
        predictions = jnp.argmax(logits, axis=-1)
    accuracy = jnp.mean(predictions == y)
    return accuracy, predictions

# 4. Training loop
log("Starting training...")
losses  = []
epochs = 10
for epoch in range(epochs):
    epoch_loss = 0
    count = 0
    for i, (x,y)  in enumerate(train_data):
        params, loss = train_step(params, x, y)
        epoch_loss += jax.numpy.abs(loss)
        count += 1
    
    avg_loss = epoch_loss / count
    losses.append(avg_loss)
    log(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.6f}")
        
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

# 5. Testing loop
log("Starting evaluation on test set...")
total_accuracy = 0
num_batches = 0
all_images = []
all_preds = []
all_labels = []

for i, (x, y) in enumerate(test_data):
    acc, preds = eval_step(params, x, y)
    batch_acc = float(acc)
    log(f"Batch {i+1} Test Accuracy: {batch_acc:.4f}")
    
    total_accuracy += batch_acc
    num_batches += 1
    
    # Save first batch for visualization
    if i == 0:
        all_images = x
        all_preds = preds
        all_labels = y

avg_test_accuracy = total_accuracy / num_batches
log(f"Average Test Accuracy: {avg_test_accuracy:.4f}")

# Visualize
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(min(5, len(all_images))):
    img = all_images[i]
    if img.shape[-1] == 1:
        img = img.reshape(28, 28)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"Pred: {all_preds[i]}, True: {all_labels[i]}")
    axes[i].axis('off')
plt.show()

ValueError: Can't compute input and output sizes of a 1-dimensional weights tensor. Must be at least 2D.

: 